# ChainlinkBlockRelay Test Script

Tests the ChainlinkBlockRelay deployment end-to-end:

1. **Trigger** — HTTP POST to the deployed CRE workflow
2. **Primary relay commit** — poll `committer_votes(relay, block)` on the primary chain; non-zero independent of threshold
3. **CCIP dispatch** — check `BlockHashBroadcast` event confirms the relay sent CCIP messages
4. **Target chain delivery** — check `committer_votes` on CCIP-receiving chains (re-run after ~20 min)

> **Note on threshold**: `committer_votes` succeeds as soon as *this* relay commits, regardless of
> how many other committers (LZBlockRelay, multisig) have committed. `last_confirmed_block_number`
> only advances once `threshold` is reached across all committers.

## 1. Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

NETWORK_TYPE = "testnets"  # "testnets" or "mainnets"

# Chain where the CRE workflow delivers onReport
CRE_RELAY_CHAIN = "base-sepolia"

# Chains expected to receive the block via CCIP broadcast from the primary relay.
# Must have ChainlinkBlockRelay deployed and configured as a CCIP peer of CRE_RELAY_CHAIN.
CCIP_TARGET_CHAINS = ["optimism-sepolia"]

# Broadcast gas limits — used to build the CRE workflow input in §7.
CCIP_RECEIVE_GAS_LIMIT = 150_000  # gas for ccipReceive on each target chain
ON_REPORT_GAS_LIMIT = 1_000_000  # gas for onReport on the primary relay
# Multiplier applied to on-chain fee quotes for headroom (fees can drift before the
# workflow executes, and the relay must hold enough balance to cover them).
CCIP_FEE_BUFFER = 1.2

# The relay pays CCIP fees from its own balance during onReport (the CRE forwarder does
# not attach value). If FUND_RELAY is True and the relay balance can't cover the quoted
# fees, §7 transfers the shortfall from your account (WEB3_TESTNET_PK / ENCRYPTED_PK).
# If False, §7 raises when the relay is underfunded instead of triggering.
FUND_RELAY = True

# Polling: CRE is off-chain, typically processes within 30-60 s
CRE_POLL_TIMEOUT = 120  # seconds
CRE_POLL_INTERVAL = 10  # seconds

# CRE workflow selector — fill one option and leave the other blank.
# Option A: workflow ID (hex string from CRE dashboard)
CRE_WORKFLOW_ID = "00521f4a3a2f40ec36d225d76249451818f3506646aa5f68fc3e836d20f2ad5a"
# Option B: owner + name + tag (all three required if ID is empty)
CRE_WORKFLOW_OWNER = ""
CRE_WORKFLOW_NAME = ""
CRE_WORKFLOW_TAG = ""

## 2. Environment

In [ ]:
import base64
import hashlib
import json
import logging
import os
import subprocess
import sys
import time
import uuid
from pathlib import Path

import requests
from dotenv import load_dotenv
from eth_account import Account
from eth_account.messages import encode_defunct
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

sys.path.append(str(Path().resolve().parent))
from DeploymentManager import DeploymentManager
from ChainlinkMetadata import ChainlinkMetadata

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
load_dotenv()

EMPTY_BYTES32 = b"\x00" * 32

## 3. Helpers

In [ ]:
def get_vyper_abi(filepath):
    result = subprocess.run(
        ["vyper", filepath, "-f", "abi_python"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout


def poll(fn, timeout, interval, label="waiting"):
    "Call fn() every interval seconds until it returns truthy or timeout expires."
    start = time.time()
    while time.time() - start < timeout:
        result = fn()
        if result:
            return result
        print(f"  [{int(time.time() - start)}s] {label}...")
        time.sleep(interval)
    return None


# ── CRE gateway JWT helpers ───────────────────────────────────────────────────


def _b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()


def _sha256_stable(obj) -> str:
    "SHA-256 of the stable-serialized object (equivalent to json-stable-stringify)."
    s = json.dumps(obj, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(s.encode()).hexdigest()


def create_cre_jwt(jsonrpc_request: dict, private_key: str) -> str:
    "Create a CRE gateway JWT signed with an Ethereum private key."
    account = Account.from_key(private_key)
    header = {"alg": "ETH", "typ": "JWT"}
    now = int(time.time())
    payload = {
        "digest": "0x" + _sha256_stable(jsonrpc_request),
        "iss": account.address,
        "iat": now,
        "exp": now + 300,
        "jti": str(uuid.uuid4()),
    }

    encoded_header = _b64url(json.dumps(header, separators=(",", ":")).encode())
    encoded_payload = _b64url(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode())
    raw_message = f"{encoded_header}.{encoded_payload}"

    # Ethereum personal sign (adds "\x19Ethereum Signed Message:\n" prefix)
    signable = encode_defunct(text=raw_message)
    signed = account.sign_message(signable)

    r = signed.r.to_bytes(32, "big")
    s_bytes = signed.s.to_bytes(32, "big")
    recovery_id = signed.v - 27  # 0 or 1
    sig_bytes = r + s_bytes + bytes([recovery_id])

    return f"{raw_message}.{_b64url(sig_bytes)}"


def trigger_cre_workflow(
    workflow_selector: dict,
    input_data: dict,
    private_key: str,
    gateway_url: str,
):
    "POST a signed JSON-RPC 2.0 request to the CRE gateway."
    request_id = str(uuid.uuid4())
    jsonrpc_request = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "workflows.execute",
        "params": {
            "input": input_data,
            "workflow": workflow_selector,
        },
    }
    jwt = create_cre_jwt(jsonrpc_request, private_key)
    # Body must be stable-serialized — same ordering used in the JWT digest
    body = json.dumps(jsonrpc_request, sort_keys=True, separators=(",", ":"))
    return requests.post(
        gateway_url,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {jwt}"},
        data=body,
        timeout=30,
    )

## 4. Load Deployment State

In [ ]:
deployment_manager = DeploymentManager()
deployed_contracts = deployment_manager.get_all_deployed_contracts(NETWORK_TYPE)
if not deployed_contracts:
    raise ValueError(f"No deployments found for {NETWORK_TYPE}")

with open("../chain-parse/chains.json") as f:
    all_chains_config = json.load(f)[NETWORK_TYPE]

# Main chain — where MainnetBlockView lives. The CRE workflow reads its source block
# from here, so this (not CRE_RELAY_CHAIN) is the chain baseline_eth_block must come from.
main_chain = next((c for c, cfg in all_chains_config.items() if cfg.get("is_main_chain")), None)
if not main_chain:
    raise ValueError(f"No main chain defined for {NETWORK_TYPE}")
logging.info(f"Main chain: {main_chain}")

all_test_chains = [CRE_RELAY_CHAIN] + CCIP_TARGET_CHAINS
for chain in all_test_chains:
    contracts = deployed_contracts.get(chain, {})
    if not contracts:
        logging.warning(f"{chain}: no deployments found")
    elif "ChainlinkBlockRelay" not in contracts:
        logging.warning(f"{chain}: ChainlinkBlockRelay not deployed — check deploy.ipynb")
    else:
        logging.info(f"{chain}: ChainlinkBlockRelay at {contracts['ChainlinkBlockRelay']}")

## 5. Initialize Chain State

In [ ]:
ABI_RELAY = get_vyper_abi("../../contracts/messengers/ChainlinkBlockRelay.vy")
ABI_ORACLE = get_vyper_abi("../../contracts/BlockOracle.vy")
ABI_VIEW = get_vyper_abi("../../contracts/MainnetBlockView.vy")

ankr_key = os.environ.get("ANKR_API_KEY")
drpc_key = os.environ.get("DRPC_API_KEY")
state_dict = {}

# Connect to every relay/target chain plus the main chain (source of the relayed block).
# dict.fromkeys dedupes while preserving order, in case main_chain == CRE_RELAY_CHAIN.
chains_to_connect = list(dict.fromkeys([main_chain] + all_test_chains))

for chain_name in chains_to_connect:
    config = all_chains_config.get(chain_name)
    if not config:
        logging.warning(f"No chain config for {chain_name}")
        continue

    rpc_url = None
    for rpc_type in ["ankr", "drpc", "public"]:
        if rpc_type == "public" and NETWORK_TYPE == "testnets":
            rpc_type = "rpc"
        val = config.get(rpc_type)
        if val:
            if rpc_type == "ankr":
                rpc_url = val.format(ankr_key)
            elif rpc_type == "drpc":
                rpc_url = val.format(drpc_key)
            else:
                rpc_url = val
            break

    if not rpc_url:
        logging.warning(f"No RPC for {chain_name}")
        continue

    w3 = Web3(Web3.HTTPProvider(rpc_url))
    w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
    contracts = deployed_contracts.get(chain_name, {})

    state_dict[chain_name] = {"w3": w3}

    if "MainnetBlockView" in contracts:
        state_dict[chain_name]["view_w3"] = w3.eth.contract(
            address=contracts["MainnetBlockView"], abi=ABI_VIEW
        )
        state_dict[chain_name]["view_address"] = contracts["MainnetBlockView"]

    if "BlockOracle" in contracts:
        state_dict[chain_name]["oracle_w3"] = w3.eth.contract(
            address=contracts["BlockOracle"], abi=ABI_ORACLE
        )
        state_dict[chain_name]["oracle_address"] = contracts["BlockOracle"]

    if "ChainlinkBlockRelay" in contracts:
        state_dict[chain_name]["relay_w3"] = w3.eth.contract(
            address=contracts["ChainlinkBlockRelay"], abi=ABI_RELAY
        )
        state_dict[chain_name]["relay_address"] = contracts["ChainlinkBlockRelay"]

    logging.info(
        f"{chain_name}: connected"
        f" | view={'view_w3' in state_dict[chain_name]}"
        f" | oracle={'oracle_w3' in state_dict[chain_name]}"
        f" | relay={'relay_w3' in state_dict[chain_name]}"
    )

## 6. Baseline Oracle State

In [ ]:
# The block being committed must come from MainnetBlockView on the MAIN chain (not
# CRE_RELAY_CHAIN's own head) — poll the view contract directly and pass its exact
# block number to the CRE workflow (§7), so §8 can check that one block instead of a range.
view_contract = state_dict[main_chain].get("view_w3")
if view_contract is None:
    raise ValueError(
        f"MainnetBlockView not deployed on main chain {main_chain} — check deploy.ipynb"
    )

baseline_eth_block, baseline_eth_hash = view_contract.functions.get_blockhash().call()
print(f"MainnetBlockView ({main_chain}) baseline block: {baseline_eth_block}")
print(f"  hash: 0x{baseline_eth_hash.hex()}\n")

print("Current oracle state:")
for chain_name in all_test_chains:
    s = state_dict.get(chain_name, {})
    if "oracle_w3" not in s:
        print(f"  {chain_name}: oracle not loaded")
        continue
    oracle = s["oracle_w3"]
    relay_addr = s.get("relay_address")
    last = oracle.functions.last_confirmed_block_number().call()
    vote = (
        oracle.functions.committer_votes(relay_addr, last).call()
        if relay_addr and last
        else EMPTY_BYTES32
    )
    print(
        f"  {chain_name}: last_confirmed={last}"
        f" | relay_vote={'set' if vote != EMPTY_BYTES32 else 'none'}"
    )

## 7. Trigger CRE Workflow

In [ ]:
CRE_GATEWAY_URL = os.environ.get("CRE_GATEWAY_URL")
CRE_PRIVATE_KEY = os.environ.get("CRE_PRIVATE_KEY")

if not CRE_GATEWAY_URL:
    raise ValueError(
        "CRE_GATEWAY_URL not set — copy the gateway URL from the CRE workflow dashboard"
    )
if not CRE_PRIVATE_KEY:
    raise ValueError(
        "CRE_PRIVATE_KEY not set — use the private key authorised to trigger this workflow"
    )

# Build workflow selector from config vars (§1)
if CRE_WORKFLOW_ID:
    workflow_selector = {"workflowID": CRE_WORKFLOW_ID}
elif CRE_WORKFLOW_OWNER and CRE_WORKFLOW_NAME and CRE_WORKFLOW_TAG:
    workflow_selector = {
        "workflowOwner": CRE_WORKFLOW_OWNER,
        "workflowName": CRE_WORKFLOW_NAME,
        "workflowTag": CRE_WORKFLOW_TAG,
    }
else:
    raise ValueError(
        "No workflow selector — set CRE_WORKFLOW_ID, or all three of "
        "CRE_WORKFLOW_OWNER + CRE_WORKFLOW_NAME + CRE_WORKFLOW_TAG in §1"
    )

# ── Build the workflow input (RequestPayload) ─────────────────────────────────
# Schema (cre-workflow/workflow/types/types.ts):
#   RequestPayload   = { blockNumber?, data: BroadcastPayload[] }
#   BroadcastPayload = { relay:{chainSelectorName, contractAddress},
#                        targetChains:[{selector, fees}], ccipReceiveGasLimit, onReportGasLimit }
# All numeric fields are strings. Fees are quoted on-chain from the primary relay;
# gas limits come from the §1 constants.
cl = ChainlinkMetadata()

w3_primary = state_dict[CRE_RELAY_CHAIN]["w3"]
primary_relay_w3 = state_dict[CRE_RELAY_CHAIN]["relay_w3"]
primary_relay_addr = state_dict[CRE_RELAY_CHAIN]["relay_address"]
relay_cl_name = cl.get_chain_metadata(CRE_RELAY_CHAIN, NETWORK_TYPE)["cl_chain_name"]

# CCIP selectors of the broadcast target chains
target_selectors = [
    cl.get_chain_metadata(chain_name, NETWORK_TYPE)["chain_selector"]
    for chain_name in CCIP_TARGET_CHAINS
]

# Quote per-target CCIP fees from the primary relay (0 => peer not configured there)
if target_selectors:
    quoted_fees = primary_relay_w3.functions.quote_broadcast_fees(
        target_selectors, CCIP_RECEIVE_GAS_LIMIT
    ).call()
else:
    quoted_fees = []

target_chains = []
for chain_name, selector, fee in zip(CCIP_TARGET_CHAINS, target_selectors, quoted_fees):
    if fee == 0:
        logging.warning(
            f"Quoted fee 0 for {chain_name} (selector {selector}) — "
            f"CCIP peer likely not configured on {CRE_RELAY_CHAIN} (run config.ipynb §7b)"
        )
    buffered_fee = int(fee * CCIP_FEE_BUFFER)
    target_chains.append({"selector": str(selector), "fees": str(buffered_fee)})
    logging.info(f"  target {chain_name}: selector {selector}, fee {buffered_fee / 1e18:.9f} ETH")

input_data = {
    # Pins the workflow to the exact block polled from MainnetBlockView in §6, instead
    # of letting it read whatever block is latest at execution time.
    "blockNumber": str(baseline_eth_block),
    "data": [
        {
            "relay": {
                "chainSelectorName": relay_cl_name,
                "contractAddress": primary_relay_addr,
            },
            "targetChains": target_chains,
            "ccipReceiveGasLimit": str(CCIP_RECEIVE_GAS_LIMIT),
            "onReportGasLimit": str(ON_REPORT_GAS_LIMIT),
        }
    ],
}

# ── Pre-flight: ensure the primary relay can cover the CCIP fees ──────────────
# onReport spends fees from the relay's own balance (assert self.balance >= total_fee).
total_fees = sum(int(t["fees"]) for t in target_chains)
relay_balance = w3_primary.eth.get_balance(primary_relay_addr)
logging.info(
    f"Primary relay balance {relay_balance / 1e18:.9f} ETH | "
    f"required fees {total_fees / 1e18:.9f} ETH"
)

if total_fees > 0 and relay_balance < total_fees:
    shortfall = total_fees - relay_balance
    if not FUND_RELAY:
        raise ValueError(
            f"Primary relay {primary_relay_addr} underfunded by {shortfall / 1e18:.9f} ETH "
            f"(has {relay_balance / 1e18:.9f}, needs {total_fees / 1e18:.9f}). "
            f"Set FUND_RELAY=True in §1 to auto-fund, or top it up manually."
        )

    # Resolve a funding account (same key sources as deploy/config notebooks)
    if NETWORK_TYPE == "testnets":
        funding_pk = os.environ.get("WEB3_TESTNET_PK")
        if not funding_pk:
            raise ValueError("WEB3_TESTNET_PK not set — required to fund the relay")
        funding_account = Account.from_key(funding_pk)
    else:
        sys.path.append(os.path.expanduser("~/projects/keys/scripts"))
        from secure_key_utils import get_web3_account
        from getpass import getpass

        funding_account = get_web3_account(os.environ.get("ENCRYPTED_PK"), getpass())

    logging.info(
        f"FUND_RELAY: sending {shortfall / 1e18:.9f} ETH "
        f"from {funding_account.address} to relay {primary_relay_addr}..."
    )
    fund_tx = {
        "from": funding_account.address,
        "to": primary_relay_addr,
        "value": shortfall,
        "nonce": w3_primary.eth.get_transaction_count(funding_account.address),
        "chainId": w3_primary.eth.chain_id,
        "gasPrice": int(1.2 * w3_primary.eth.gas_price),
    }
    fund_tx["gas"] = int(w3_primary.eth.estimate_gas(fund_tx) * 1.2)
    signed_fund = w3_primary.eth.account.sign_transaction(fund_tx, funding_account.key)
    fund_tx_hash = w3_primary.eth.send_raw_transaction(signed_fund.raw_transaction)
    fund_receipt = w3_primary.eth.wait_for_transaction_receipt(fund_tx_hash)
    if fund_receipt.status != 1:
        raise Exception(f"Funding tx failed: 0x{fund_tx_hash.hex()}")
    relay_balance = w3_primary.eth.get_balance(primary_relay_addr)
    logging.info(
        f"Funded relay — new balance {relay_balance / 1e18:.9f} ETH (tx 0x{fund_tx_hash.hex()})"
    )

_account = Account.from_key(CRE_PRIVATE_KEY)
logging.info(f"Triggering CRE workflow as {_account.address}")
logging.info(f"  Gateway:  {CRE_GATEWAY_URL}")
logging.info(f"  Selector: {workflow_selector}")
logging.info(f"  Input:    {json.dumps(input_data)}")

resp = trigger_cre_workflow(workflow_selector, input_data, CRE_PRIVATE_KEY, CRE_GATEWAY_URL)
resp.raise_for_status()
trigger_time = time.time()
logging.info(f"CRE workflow triggered for ETH block {baseline_eth_block}: HTTP {resp.status_code}")

## 8. Verify Primary Relay Committed

In [ ]:
oracle = state_dict[CRE_RELAY_CHAIN]["oracle_w3"]
relay_addr = state_dict[CRE_RELAY_CHAIN]["relay_address"]

# The workflow was triggered with an explicit blockNumber (§7), so it must commit
# exactly baseline_eth_block — no need to scan a range.


def check_relay_committed():
    h = oracle.functions.committer_votes(relay_addr, baseline_eth_block).call()
    return (baseline_eth_block, h) if h != EMPTY_BYTES32 else None


print(
    f"Polling committer_votes({baseline_eth_block}) on {CRE_RELAY_CHAIN} (timeout {CRE_POLL_TIMEOUT}s)..."
)
result = poll(
    check_relay_committed, CRE_POLL_TIMEOUT, CRE_POLL_INTERVAL, "waiting for relay commit"
)

if result:
    committed_block, committed_hash = result
    print(f"\n✓ ChainlinkBlockRelay committed block {committed_block}: 0x{committed_hash.hex()}")
else:
    committed_block, committed_hash = None, None
    print(f"\n✗ No commit for block {baseline_eth_block} within {CRE_POLL_TIMEOUT}s")

## 9. Verify CCIP Broadcast Dispatch

In [ ]:
# BlockHashBroadcast is emitted by ChainlinkBlockRelay when it sends CCIP messages.
# This confirms dispatch without waiting for cross-chain delivery.
if not committed_block:
    print("Skipped — no committed block from §8")
else:
    relay_w3 = state_dict[CRE_RELAY_CHAIN]["relay_w3"]
    # Scan range is on CRE_RELAY_CHAIN itself, so use its own block height —
    # not eth_w3 (main_chain), whose block numbers aren't comparable across chains.
    scan_from = max(0, state_dict[CRE_RELAY_CHAIN]["w3"].eth.block_number - 200)

    try:
        broadcast_logs = relay_w3.events.BlockHashBroadcast.create_filter(
            fromBlock=hex(scan_from)
        ).get_all_entries()

        matching = [log for log in broadcast_logs if log["args"]["block_number"] == committed_block]

        if matching:
            targets = matching[0]["args"]["targets"]
            selectors = [t["chain_selector"] for t in targets]
            print(f"✓ BlockHashBroadcast: CCIP sent to {len(selectors)} chain(s)")
            for t in targets:
                print(f"  selector {t['chain_selector']}, fee {t['fee'] / 1e18:.6f} ETH")
        else:
            print("⚠ BlockHashBroadcast not found for this block")
            print("  Possible causes: no CCIP peers configured, or broadcast was skipped")
    except Exception as e:
        logging.warning(f"Could not fetch BlockHashBroadcast logs: {e}")

    # Oracle threshold check
    confirmed_hash = oracle.functions.get_block_hash(committed_block).call()
    if confirmed_hash != EMPTY_BYTES32:
        print(f"\n✓ Block {committed_block} already confirmed in oracle (threshold reached)")
    else:
        count = oracle.functions.commitment_count(committed_block, committed_hash).call()
        threshold = oracle.functions.threshold().call()
        print(f"\n  Oracle commitments: {count}/{threshold} — threshold not yet reached")
        print("  Expected: other committers (LZBlockRelay, multisig backup) must also commit")

## 10. Verify Target Chain Delivery

CCIP delivery on mainnet takes ~10-20 minutes. Re-run this cell to check progress.

In [ ]:
if not committed_block:
    print("Skipped — no committed block from §8")
else:
    print(f"Checking CCIP delivery for block {committed_block}:\n")
    for chain_name in CCIP_TARGET_CHAINS:
        s = state_dict.get(chain_name, {})
        if "oracle_w3" not in s or "relay_address" not in s:
            print(f"  {chain_name}: not initialized — check §5")
            continue

        target_oracle = s["oracle_w3"]
        target_relay = s["relay_address"]

        vote = target_oracle.functions.committer_votes(target_relay, committed_block).call()
        if vote != EMPTY_BYTES32:
            confirmed = target_oracle.functions.get_block_hash(committed_block).call()
            print(
                f"  ✓ {chain_name}: committed 0x{vote.hex()[:16]}..."
                f" | oracle confirmed: {confirmed != EMPTY_BYTES32}"
            )
        else:
            print(f"  ⏳ {chain_name}: not yet delivered — re-run this cell in a few minutes")

## 11. Summary

In [ ]:
print("\n" + "=" * 72)
print("CHAINLINK RELAY TEST SUMMARY")
print("=" * 72)
print(f"Network:       {NETWORK_TYPE}")
print(f"Primary chain: {CRE_RELAY_CHAIN}")
print(f"Main chain:    {main_chain}")
print(f"ETH baseline:  block {baseline_eth_block} (on {main_chain})")
print()

if committed_block:
    print(f"✓ CRE workflow delivered block {committed_block} to ChainlinkBlockRelay")
    print(f"  Hash: 0x{committed_hash.hex()}")

    confirmed_hash = oracle.functions.get_block_hash(committed_block).call()
    count = oracle.functions.commitment_count(committed_block, committed_hash).call()
    threshold = oracle.functions.threshold().call()
    status = (
        "CONFIRMED" if confirmed_hash != EMPTY_BYTES32 else f"{count}/{threshold} commits (pending)"
    )
    print(f"  Oracle:  {status}")

    print()
    for chain_name in CCIP_TARGET_CHAINS:
        s = state_dict.get(chain_name, {})
        if "oracle_w3" not in s or "relay_address" not in s:
            print(f"  {chain_name}: not loaded")
            continue
        vote = s["oracle_w3"].functions.committer_votes(s["relay_address"], committed_block).call()
        symbol = "✓" if vote != EMPTY_BYTES32 else "⏳"
        label = "committed via CCIP" if vote != EMPTY_BYTES32 else "pending CCIP delivery"
        print(f"  {symbol} {chain_name}: {label}")
else:
    print("✗ ChainlinkBlockRelay commit was not detected")
    print(
        "  Check: CRE_GATEWAY_URL is correct, CRE_PRIVATE_KEY is authorized, workflow is deployed, relay is configured"
    )